# Notebook 3: Inter-Event Time Regime Classification


In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.interevent import compute_interevent_times, fit_distributions, classify_regime
from src.spatial import assign_cells
from src.gutenberg_richter import compute_bvalue_grid
from src.plotting import setup_style, save_figure, plot_regime_map

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
print(f"Loaded {len(df):,} events")


Loaded 681,450 events


## 3.1 Spatial Gridding and IET Computation


In [3]:
df_cells = assign_cells(df, cell_size=2.0)

results = []
example_cells = {"poisson": None, "clustered": None, "quasi_periodic": None, "ambiguous": None}

for (clat, clon), grp in df_cells.groupby(["cell_lat", "cell_lon"]):
    if len(grp) < 100:
        continue

    times_sorted = grp.sort_values("time")["time"].values
    iet = compute_interevent_times(times_sorted)
    iet_hours = iet / 3600.0  # Convert to hours
    iet_hours = iet_hours[iet_hours > 0]

    if len(iet_hours) < 50:
        continue

    median_iet_days = np.median(iet_hours) / 24.0
    if median_iet_days > 30:
        continue

    try:
        fit_result = fit_distributions(iet_hours)
        regime = classify_regime(fit_result)
    except Exception:
        continue

    # Get Weibull k if available
    weibull_k = fit_result["weibull"]["params"][0]

    results.append({
        "cell_lat": clat,
        "cell_lon": clon,
        "regime": regime,
        "best_aic": fit_result["best_aic"],
        "weibull_k": weibull_k,
        "n_events": len(grp),
        "median_iet_hours": np.median(iet_hours),
    })

    # Save example cells
    if example_cells[regime] is None:
        example_cells[regime] = {"iet_hours": iet_hours, "fit_result": fit_result,
                                  "lat": clat, "lon": clon, "n": len(grp)}

regime_df = pd.DataFrame(results)
print(f"Classified {len(regime_df)} cells")
print(regime_df["regime"].value_counts())


Classified 715 cells
regime
clustered    650
ambiguous     53
poisson       12
Name: count, dtype: int64


## 3.2 Global Regime Map


In [4]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_regime_map(regime_df["cell_lat"], regime_df["cell_lon"], regime_df["regime"], ax=ax)
save_figure(fig, "03_regime_classification_map")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14638/3266278282.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.3 Weibull k Map


In [5]:
from src.plotting import plot_global_map

fig, ax = plt.subplots(figsize=(14, 7))
plot_global_map(
    regime_df["cell_lat"], regime_df["cell_lon"], regime_df["weibull_k"],
    cmap="coolwarm", vmin=0.5, vmax=1.5,
    label="Weibull shape parameter k",
    title="Weibull k Parameter (k<1: clustered, k=1: Poisson, k>1: quasi-periodic)",
    ax=ax
)
save_figure(fig, "03_weibull_k_map")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14638/936925768.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.4 Example Distributions


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes_flat = axes.flatten()

for i, (regime_name, label) in enumerate([
    ("poisson", "Poisson-like"), ("clustered", "Clustered-like"),
    ("quasi_periodic", "Quasi-periodic-like"), ("ambiguous", "Ambiguous")
]):
    ax = axes_flat[i]
    ex = example_cells.get(regime_name)
    if ex is None:
        ax.set_title(f"{label} — no example found")
        continue

    iet = ex["iet_hours"]
    fit = ex["fit_result"]

    # Empirical CDF
    sorted_iet = np.sort(iet)
    ecdf = np.arange(1, len(sorted_iet) + 1) / len(sorted_iet)
    ax.plot(sorted_iet, ecdf, "k-", linewidth=1.5, label="Empirical CDF")

    # Fitted CDFs
    x = np.linspace(sorted_iet[0], np.percentile(sorted_iet, 99), 200)
    colors = {"exponential": "#457B9D", "gamma": "#E63946",
              "lognormal": "#F4A261", "weibull": "#2A9D8F"}
    for dist_name, color in colors.items():
        params = fit[dist_name]["params"]
        dist = getattr(stats, "expon" if dist_name == "exponential" else
                       "lognorm" if dist_name == "lognormal" else
                       "weibull_min" if dist_name == "weibull" else "gamma")
        cdf_vals = dist.cdf(x, *params)
        aic = fit[dist_name]["aic"]
        ax.plot(x, cdf_vals, color=color, linewidth=1, alpha=0.8,
                label=f"{dist_name} (AIC={aic:.0f})")

    ax.set_xlabel("Inter-event time (hours)")
    ax.set_ylabel("CDF")
    ax.set_title(f"{label} (lat={ex['lat']:.0f}, lon={ex['lon']:.0f})")
    ax.legend(fontsize=7)
    ax.set_xscale("log")

fig.suptitle("Example IET Distributions by Regime", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "03_example_distributions")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14638/2259060495.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.5 Weibull k vs. b-value


In [7]:
bvalue_grid = compute_bvalue_grid(df, cell_size=2.0, min_events=50)
merged = regime_df.merge(bvalue_grid[["cell_lat", "cell_lon", "b_value"]],
                         on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
color_map = {"poisson": "#457B9D", "clustered": "#E63946",
             "quasi_periodic": "#2A9D8F", "ambiguous": "#AAAAAA"}
for regime_name in color_map:
    subset = merged[merged["regime"] == regime_name]
    ax.scatter(subset["b_value"], subset["weibull_k"], c=color_map[regime_name],
               s=20, alpha=0.5, label=regime_name.replace("_", " ").title())

ax.set_xlabel("b-value")
ax.set_ylabel("Weibull k")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Weibull Shape Parameter vs. b-value")
ax.legend()
save_figure(fig, "03_k_vs_bvalue")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14638/3514603236.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.6 Weibull k vs. Depth


In [8]:
df_cells_depth = assign_cells(df, cell_size=2.0)
depth_med = df_cells_depth.groupby(["cell_lat", "cell_lon"])["depth"].median().reset_index()
depth_med.columns = ["cell_lat", "cell_lon", "median_depth"]

k_depth = regime_df.merge(depth_med, on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(k_depth["median_depth"], k_depth["weibull_k"],
                c=k_depth["n_events"], cmap="viridis", s=15, alpha=0.5,
                norm=plt.matplotlib.colors.LogNorm())
plt.colorbar(sc, ax=ax, label="Number of events")
ax.set_xlabel("Median Depth (km)")
ax.set_ylabel("Weibull k")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Weibull k vs. Predominant Event Depth")
save_figure(fig, "03_k_vs_depth")
plt.show()


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_14638/3159379449.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook classified seismically active 2°×2° grid cells globally into four inter-event time regimes:

- **Poisson-like** (Weibull k ≈ 1): Events arrive randomly and independently, consistent with a memoryless process. Typical of stable tectonic regions with background seismicity.
- **Clustered-like** (Weibull k < 1): Short inter-event times are over-represented, indicating aftershock-dominated or swarm-like behavior.
- **Quasi-periodic-like** (Weibull k > 1): Inter-event times are more regular than random, suggesting stress-loading cycles or repeating earthquake sequences.
- **Ambiguous**: No single distribution fits significantly better than the others.

Key findings:
- The global regime map reveals spatial coherence: subduction zones and active fault systems tend toward clustered behavior, while intraplate regions lean Poisson-like.
- The Weibull k parameter provides a continuous measure of temporal clustering that correlates with tectonic setting.
- Comparison with b-values shows that regions with low b-values (dominated by larger events) tend to exhibit different temporal patterns than high-b-value regions.
- Depth dependence of Weibull k suggests that shallow seismicity is more clustered, consistent with aftershock productivity scaling with depth.
